# 1 · Juros altos se justificam?

**Pergunta:** o patamar de juros contemporâneo é explicado por inflação, atividade
e emprego — ou há um componente institucional a mais do que essas variáveis pedem?

**O que argumentar de forma defensável exige mostrar, com dados:**
1. que o ciclo da Selic acompanha (com defasagem) a inflação realizada **e esperada** (Focus);
2. o histórico e o patamar atual do **juro real ex ante** (Selic − inflação esperada), e como
   ele se compara à média histórica;
3. se atividade (IBC-Br) e desemprego explicam a direção;
4. onde a Selic fica em relação a uma **regra de Taylor** simples — referência explícita de
   "o quanto é mais do que uma regra básica pediria" (não é "o juro correto").

**Fontes:** SGS 432/4189/13522/24364; SIDRA 6381 v4099 (desemprego, trimestre móvel);
Focus/Olinda (`ExpectativasMercadoInflacao12Meses`, média não suavizada).


In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "src").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

from src.viz.graficos import (
    estilo, salvar, AZUL, VERMELHO, VERDE, LARANJA, ROXO, CINZA, CORES_DETENTOR,
)
from src.processa import indicadores as ind

estilo()
print("diretório de dados:", p / "dados" / "processado")


diretório de dados: /home/vanessa/devel/juros_e_divida/dados/processado


In [2]:
m = ind.selic_real_mensal()
m = m.drop(columns=["selic_mensal"])
print(m.tail(3).round(2).to_string())
print(f"\nperíodo: {m.index.min().date()} → {m.index.max().date()}")

            selic_meta  ipca_12m  focus_ipca_12m  ibcbr_var12m  desemprego  juro_real_ex_post  juro_real_ex_ante
2026-09-30       13.75      4.58            4.81           NaN        5.20               9.20               8.97
2026-10-31       13.75       NaN            4.90           NaN         NaN                NaN               8.75
2026-11-30       13.75       NaN             NaN           NaN         NaN                NaN                NaN

período: 2002-01-31 → 2026-11-30


In [3]:
fig, ax = plt.subplots(figsize=(11, 5))
m["selic_meta"].plot(ax=ax, color=AZUL, label="Selic meta (fim de mês)")
m["ipca_12m"].dropna().plot(ax=ax, color=VERMELHO, label="IPCA 12m (realizado)")
m["focus_ipca_12m"].dropna().plot(ax=ax, color=LARANJA, ls="--", label="Focus: IPCA 12m (média)")
ax.axhline(4.5, color=CINZA, lw=1, ls=":", alpha=0.7)
ax.text(pd.Timestamp("2002-06-01"), 5.2, "meta de inflação (4,5%)", fontsize=9, color=CINZA)
ax.set_ylabel("% a.a.")
ax.set_title("Selic meta vs inflação realizada e esperada (2002–2026)")
ax.legend()
salvar(fig, "01_selic_vs_inflacao")

fig -> viz/figs/01_selic_vs_inflacao.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_selic_vs_inflacao.png')

In [4]:
fig, ax = plt.subplots(figsize=(11, 5))
m["juro_real_ex_ante"].dropna().plot(ax=ax, color=VERDE, label="juro real ex ante (Selic − Focus)")
m["juro_real_ex_post"].dropna().plot(ax=ax, color=CINZA, label="juro real ex post (Selic − IPCA)")
ax.axhline(0, color="k", lw=1)
media_ant = m.loc[: "2014", "juro_real_ex_ante"].mean()
ax.axhline(media_ant, color=VERDE, ls=":", lw=1.2)
ax.text(pd.Timestamp("2002-06-01"), media_ant + 0.4, f"média 2002–2014: {media_ant:.1f}%", fontsize=9, color=VERDE)
ax.set_ylabel("% a.a.")
ax.set_title("Juro real brasileiro: ex ante vs ex post")
ax.legend()
salvar(fig, "01_juro_real")

fig -> viz/figs/01_juro_real.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_juro_real.png')

In [5]:
janelas = [
    ("2002-2014", slice("2002", "2014")),
    ("2015-2019", slice("2015", "2019")),
    ("2020-2021 (juros a 2%)", slice("2020", "2021")),
    ("2022-2023 (aperto pós-pandemia)", slice("2022", "2023")),
    ("2024-2026 (patamar atual)", slice("2024", "2026")),
]
linhas = []
for rotulo, jan in janelas:
    rec = m.loc[jan]
    linhas.append({
        "período": rotulo,
        "selic média": rec.selic_meta.mean(),
        "juro real ex ante médio": rec.juro_real_ex_ante.mean(),
        "desemprego médio": rec.desemprego.mean(),
    })
est = pd.DataFrame(linhas).set_index("período")
print(est.round(1).to_string())

                                 selic média  juro real ex ante médio  desemprego médio
período                                                                                
2002-2014                              13.60                     8.10              7.10
2015-2019                              10.00                     5.10             11.40
2020-2021 (juros a 2%)                  3.80                    -0.40             13.50
2022-2023 (aperto pós-pandemia)        12.90                     7.80              8.80
2024-2026 (patamar atual)              13.20                     8.80              6.20


In [6]:
fig, ax = plt.subplots(figsize=(11, 5))
ax2 = ax.twinx()
m["desemprego"].dropna().plot(ax=ax, color=ROXO, label="desemprego (3m móvel, eixo esq.)")
m["ibcbr_var12m"].dropna().plot(ax=ax2, color=AZUL, label="IBC-Br var 12m % (eixo dir.)")
m["juro_real_ex_ante"].dropna().plot(ax=ax, color=VERMELHO, alpha=0.45, label="juro real ex ante")
ax2.set_ylim(-15, 15)
ax.set_ylabel("%")
ax2.set_ylabel("%")
ax.set_title("Juro real vs atividade e emprego")
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=9)
salvar(fig, "01_juro_real_vs_atividade")

fig -> viz/figs/01_juro_real_vs_atividade.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_juro_real_vs_atividade.png')

In [7]:
# Regra de Taylor simples (r* = 2,5%; meta 4,5%; peso 0,5) — referência, não "juro correto"
t = m[["selic_meta", "focus_ipca_12m"]].dropna()
t["taylor"] = 2.5 + t["focus_ipca_12m"] + 0.5 * (t["focus_ipca_12m"] - 4.5)
t["desvio"] = t["selic_meta"] - t["taylor"]

print("Selic vs regra de Taylor (fim de ano):")
print(t.resample("YE").last().round(1).to_string())
print("Desvio médio (Selic − Taylor) por período:")
for nome, jan in [("2003-14", slice("2003", "2014")), ("2015-19", slice("2015", "2019")),
                  ("2019-21", slice("2019", "2021")), ("2022-23", slice("2022", "2023")),
                  ("2024-26", slice("2024", "2026"))]:
    print(f"  {nome}: {t.loc[jan, 'desvio'].mean():.1f} p.p.")

Selic vs regra de Taylor (fim de ano):
            selic_meta  focus_ipca_12m  taylor  desvio
2002-12-31       25.00           13.20   20.10    4.90
2003-12-31       16.50            5.90    9.10    7.40
2004-12-31       17.80            6.10    9.40    8.40
2005-12-31       18.00            4.50    6.90   11.10
2006-12-31       13.20            4.10    6.40    6.90
2007-12-31       11.20            4.40    6.90    4.30
2008-12-31       13.80            4.90    7.60    6.10
2009-12-31        8.80            4.40    6.80    1.90
2010-12-31       10.80            5.50    8.50    2.20
2011-12-31       11.00            5.40    8.30    2.70
2012-12-31        7.20            5.60    8.70   -1.40
2013-12-31       10.00            6.10    9.40    0.60
2014-12-31       11.80            6.70   10.30    1.40
2015-12-31       14.20            7.20   11.00    3.20
2016-12-31       13.80            4.70    7.30    6.50
2017-12-31        7.00            3.80    5.90    1.10
2018-12-31        6.50    

In [8]:
fig, ax = plt.subplots(figsize=(11, 5))
t["selic_meta"].plot(ax=ax, color=AZUL, label="Selic meta")
t["taylor"].plot(ax=ax, color=CINZA, ls="--", label="Regra de Taylor simples (r*=2,5; meta 4,5)")
t["desvio"].plot(ax=ax, color=VERMELHO, alpha=0.4, label="desvio (Selic − Taylor)")
ax.axhline(t["desvio"].mean(), color=VERMELHO, ls=":", lw=1)
ax.set_ylabel("% a.a.")
ax.set_title("O que a regra de Taylor pediria e onde a Selic ficou")
ax.legend(fontsize=9)
salvar(fig, "01_taylor")

fig -> viz/figs/01_taylor.png


PosixPath('/home/vanessa/devel/juros_e_divida/viz/figs/01_taylor.png')

**Achados verificáveis (P1).**

- A Selic acompanha a inflação **esperada** (Focus) com defasagem de poucos
  meses — a *direção* do ciclo não é arbitrária. O que se destaca é a
  **magnitude**: com Focus em ~4,8–5,0% e Selic em 13–15% a.a., o juro real
  ex ante de 2024–2026 tem **média de ~8,8 p.p.** (6,9 em 2024 → 9,6 em 2025
  → 10,1 em 2026) e **pico de 11,0 p.p. em fev/2026** — a média mais alta
  desde 2016. Níveis do mesmo porte já ocorreram em outras situações:
  inflação em dois dígitos (2003–04), ajuste pós-choque (2015–16), picos de
  crise (2008) e o aperto pós-pandemia (2022–23). E hoje ele coexiste com
  inflação de ~4,6% — dentro da meta.
- A regra de Taylor simples pediria Selic de ~7,5–9% a.a. para inflação
  esperada de ~5%. O **desvio médio atual (2024–26: ~6,5 p.p.) é o maior
  entre as janelas comparadas** (2003–14: ~4,9; 2022–23: ~5,2 p.p.) — mas
  **não é recorde histórico**: a MM36 chegou a 9,9 p.p. em fev/2006. Nos
  dois primeiros mandatos Lula o
  excesso tinha contrapeso: Focus de 10–14% (2003) e risco soberano. Hoje,
  com expectativa de ~5% e meta de 4,5%, o desvio não tem contrapeso
  inflacionário equivalente.
- Atividade e emprego não explicam o patamar: com IBC-Br de +2–3% e
  desemprego em mínimos (5,2–5,3%), a justificativa de "esfriar economia
  superaquecida" fica restrita — o aperto persiste em função de juízo
  institucional sobre credibilidade, não de variáveis observadas.

**O que isso permite concluir sem depender de documentos/declarações:** o
juro extra atual não é "preço da inflação": é um **prêmio institucional
adicional**, de magnitude sem precedente recente mesmo com inflação em
meta — quem defende que "está justificado" precisa apontar a causa
adicional (risco fiscal? credibilidade?) e ela não está nos dados
macro aqui usados.